
# An Explainable Data-Driven Digital Twin for Lithium-Ion Battery Health Monitoring

## Project Objective

This project develops a prototype Digital Twin framework for lithium-ion battery health monitoring using experimental ageing data from the NASA Battery Aging Dataset.

The framework combines:

- Experimental battery cycling data
- Discharge and impedance feature extraction
- Battery degradation and health indicator analysis
- Data-driven State-of-Health (SoH) estimation
- Hybrid prediction to handle incomplete impedance measurements
- Empirical voltage relaxation analysis
- Digital Twin state representation and updating

## Dataset

**Source:** NASA Battery Aging Dataset

**Batteries analysed:** B0005, B0006, B0007, and B0018.

The dataset contains charge, discharge, and electrochemical impedance measurements collected during battery ageing experiments.

## Research Question

To what extent can operational measurements and impedance-derived indicators improve the accuracy and interpretability of lithium-ion battery State-of-Health estimation across different battery cells?

## Modelling Approach

The project investigates three complementary approaches:

1. **Baseline model:** Ridge Regression using operational voltage, current, and temperature features.
2. **Impedance-enhanced model:** Ridge Regression incorporating impedance-related indicators (Re and Rct).
3. **Hybrid prediction strategy:** Combines the baseline and impedance-enhanced models to accommodate missing impedance observations.

Model generalisation is evaluated using Leave-One-Battery-Out (LOBO) cross-validation.

## Additional Diagnostic Analysis

Voltage relaxation characteristics and an apparent resistance indicator are investigated as potential degradation-related features. Their predictive contribution and limitations are assessed separately.

## Scope and Limitations

This implementation is a data-driven Digital Twin prototype. It does not yet constitute a validated electrochemical or physics-informed battery model. Equivalent-circuit parameter identification and physics-based degradation modelling are considered directions for future development.


## 1. Environment Setup

Import the core libraries used throughout the workflow. Additional modelling libraries are imported in the relevant analysis sections.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

## 2. Dataset Upload and Validation

The analysis uses four MATLAB files from the NASA Battery Aging Dataset: `B0005.mat`, `B0006.mat`, `B0007.mat`, and `B0018.mat`.

If the files are not already available in the current working directory, the following cell opens the Google Colab upload dialog. The workflow stops if any required file is missing.

In [ ]:
required_files = ["B0005.mat", "B0006.mat", "B0007.mat", "B0018.mat"]
missing_files = [filename for filename in required_files if not os.path.isfile(filename)]

if missing_files:
    try:
        from google.colab import files
        print("Please upload the four required NASA battery MATLAB files.")
        files.upload()
    except ImportError as exc:
        raise FileNotFoundError(
            "Required MATLAB files are missing from the working directory: "
            + ", ".join(missing_files)
        ) from exc

missing_files = [filename for filename in required_files if not os.path.isfile(filename)]
if missing_files:
    raise FileNotFoundError(
        "The following required files are still missing: "
        + ", ".join(missing_files)
    )

print("All required battery files are available:")
for filename in required_files:
    print(f"  {filename}: {os.path.getsize(filename) / 1024:.1f} KB")

## 3. Discharge Feature Extraction

Extract cycle-level voltage, current, temperature, capacity, and timestamp features from each discharge record. `record_index` is the sequential discharge-cycle index used in the temporal analyses; `matlab_record_index` preserves the original position of the record in the MATLAB cycle array.

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import loadmat


def extract_discharge_features(battery_id):

    # 1. Load MATLAB file
    mat = loadmat(
        f"{battery_id}.mat",
        squeeze_me=True,
        struct_as_record=False
    )

    battery = mat[battery_id]
    cycles = np.atleast_1d(battery.cycle)

    rows = []

    # 2. Extract discharge cycles
    for matlab_record_index, cycle in enumerate(cycles):

        if cycle.type != "discharge":
            continue

        data = cycle.data

        voltage = np.asarray(data.Voltage_measured).ravel()
        current = np.asarray(data.Current_measured).ravel()
        temperature = np.asarray(data.Temperature_measured).ravel()
        time = np.asarray(data.Time).ravel()

        capacity = float(data.Capacity)

        # 3. Convert MATLAB date into pandas Timestamp
        date_components = np.asarray(cycle.time, dtype=float).ravel()

        seconds = date_components[5]
        whole_seconds = int(np.floor(seconds))
        fractional_seconds = seconds - whole_seconds

        timestamp = pd.Timestamp(
            year=int(date_components[0]),
            month=int(date_components[1]),
            day=int(date_components[2]),
            hour=int(date_components[3]),
            minute=int(date_components[4]),
            second=whole_seconds
        ) + pd.to_timedelta(fractional_seconds, unit="s")

        # 4. Extract cycle-level features
        rows.append({

            "battery_id": battery_id,
            "record_index": len(rows) + 1,
            "matlab_record_index": matlab_record_index,
            "timestamp": timestamp,

            "capacity_Ah": capacity,

            "voltage_mean": np.mean(voltage),
            "voltage_min": np.min(voltage),
            "voltage_max": np.max(voltage),

            "current_mean": np.mean(current),

            "temperature_mean": np.mean(temperature),
            "temperature_min": np.min(temperature),
            "temperature_max": np.max(temperature),

            "temperature_rise": np.max(temperature) - np.min(temperature),

            "discharge_duration_s": time[-1] - time[0]
        })

    # 5. Create DataFrame
    df = pd.DataFrame(rows)

    # 6. Calculate SoH using the battery's initial measured capacity
    initial_capacity = df["capacity_Ah"].iloc[0]

    df["SoH_percent"] = (
        df["capacity_Ah"] / initial_capacity
    ) * 100

    return df

In [ ]:
test_df = extract_discharge_features("B0005")

print("Dataset shape:", test_df.shape)
display(test_df.head())

print("\nInitial capacity:", test_df["capacity_Ah"].iloc[0])
print("Initial SoH:", test_df["SoH_percent"].iloc[0])
print("Final SoH:", test_df["SoH_percent"].iloc[-1])

In [ ]:
battery_ids = ["B0005", "B0006", "B0007", "B0018"]

battery_datasets = {}

for battery_id in battery_ids:

    df = extract_discharge_features(battery_id)

    battery_datasets[battery_id] = df

    print(f"\n{battery_id}")
    print("Number of cycles:", len(df))
    print("Initial capacity (Ah):", round(df["capacity_Ah"].iloc[0], 4))
    print("Final capacity (Ah):", round(df["capacity_Ah"].iloc[-1], 4))
    print("Final SoH (%):", round(df["SoH_percent"].iloc[-1], 2))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 6))

for battery_id, df in battery_datasets.items():

    plt.plot(
        range(1, len(df) + 1),
        df["SoH_percent"],
        label=battery_id
    )

plt.xlabel("Discharge Cycle")
plt.ylabel("State of Health (SoH %)")
plt.title("Capacity Degradation Across Four NASA Batteries")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Impedance Feature Extraction and Timestamp Matching

Extract impedance-derived resistance indicators (`Re` and `Rct`) and associate each discharge observation with the most recent preceding impedance measurement within the specified time tolerance.

In [ ]:
def extract_impedance_features(battery_id):

    mat = loadmat(
        f"{battery_id}.mat",
        squeeze_me=True,
        struct_as_record=False
    )

    battery = mat[battery_id]
    cycles = np.atleast_1d(battery.cycle)

    rows = []

    for record_index, cycle in enumerate(cycles):

        if cycle.type != "impedance":
            continue

        data = cycle.data

        date_components = np.asarray(
            cycle.time, dtype=float
        ).ravel()

        seconds = date_components[5]
        whole_seconds = int(np.floor(seconds))
        fractional_seconds = seconds - whole_seconds

        timestamp = pd.Timestamp(
            year=int(date_components[0]),
            month=int(date_components[1]),
            day=int(date_components[2]),
            hour=int(date_components[3]),
            minute=int(date_components[4]),
            second=whole_seconds
        ) + pd.to_timedelta(fractional_seconds, unit="s")

        rows.append({

            "battery_id": battery_id,
            "impedance_record_index": record_index,
            "impedance_timestamp": timestamp,

            "Re": float(data.Re),
            "Rct": float(data.Rct)
        })

    return pd.DataFrame(rows)

In [ ]:
impedance_test = extract_impedance_features("B0005")

print("Dataset shape:", impedance_test.shape)
display(impedance_test.head())

print("\nRe statistics:")
print(impedance_test["Re"].describe())

print("\nRct statistics:")
print(impedance_test["Rct"].describe())

In [ ]:
def match_impedance_to_discharge(
    discharge_df,
    impedance_df,
    tolerance="1h"
):

    # Sort both datasets by timestamp
    discharge_sorted = discharge_df.sort_values(
        "timestamp"
    ).copy()

    impedance_sorted = impedance_df.sort_values(
        "impedance_timestamp"
    ).copy()

    # Match each discharge with its latest preceding
    # impedance measurement within the specified tolerance
    matched_df = pd.merge_asof(
        discharge_sorted,
        impedance_sorted[
            [
                "impedance_timestamp",
                "impedance_record_index",
                "Re",
                "Rct"
            ]
        ],
        left_on="timestamp",
        right_on="impedance_timestamp",
        direction="backward",
        tolerance=pd.Timedelta(tolerance)
    )

    # Calculate the age of the impedance measurement
    matched_df["impedance_age_hours"] = (
        matched_df["timestamp"]
        - matched_df["impedance_timestamp"]
    ).dt.total_seconds() / 3600

    return matched_df

In [ ]:
matched_test = match_impedance_to_discharge(
    battery_datasets["B0005"],
    impedance_test
)

print("Total discharge cycles:", len(matched_test))

print(
    "Cycles with impedance:",
    matched_test["Re"].notna().sum()
)

print(
    "Cycles without impedance:",
    matched_test["Re"].isna().sum()
)

print("\nImpedance age statistics (hours):")
print(matched_test["impedance_age_hours"].describe())

display(matched_test.head(10))

In [ ]:
impedance_datasets = {}
matched_datasets = {}

for battery_id in battery_ids:

    # Extract impedance measurements
    impedance_df = extract_impedance_features(battery_id)

    # Match them with discharge measurements
    matched_df = match_impedance_to_discharge(
        battery_datasets[battery_id],
        impedance_df,
        tolerance="1h"
    )

    impedance_datasets[battery_id] = impedance_df
    matched_datasets[battery_id] = matched_df

    print(f"\n{battery_id}")
    print("Discharge cycles:", len(matched_df))
    print("Impedance records:", len(impedance_df))
    print("Matched cycles:", matched_df["Re"].notna().sum())
    print("Unmatched cycles:", matched_df["Re"].isna().sum())
    print(
        "Matching percentage:",
        round(matched_df["Re"].notna().mean() * 100, 2),
        "%"
    )

In [ ]:
# Combine all four matched datasets
combined_df = pd.concat(
    matched_datasets.values(),
    ignore_index=True
)

# Add an explicit impedance availability indicator
combined_df["impedance_available"] = (
    combined_df["Re"].notna()
).astype(int)

# Sort by battery and timestamp
combined_df = combined_df.sort_values(
    ["battery_id", "timestamp"]
).reset_index(drop=True)

print("Combined dataset shape:", combined_df.shape)

print("\nObservations per battery:")
print(combined_df.groupby("battery_id").size())

print("\nImpedance availability:")
print(
    combined_df.groupby("battery_id")["impedance_available"]
    .agg(["sum", "count", "mean"])
)

display(combined_df.head())

In [ ]:
# Compare SoH for cycles with and without impedance measurements

availability_analysis = (
    combined_df
    .groupby(["battery_id", "impedance_available"])["SoH_percent"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

print("SoH distribution by impedance availability:")
display(availability_analysis)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, battery_id in zip(axes.flatten(), battery_ids):

    battery_data = combined_df[
        combined_df["battery_id"] == battery_id
    ]

    matched = battery_data.loc[
        battery_data["impedance_available"] == 1,
        "SoH_percent"
    ]

    unmatched = battery_data.loc[
        battery_data["impedance_available"] == 0,
        "SoH_percent"
    ]

    ax.boxplot(
        [matched, unmatched],
        tick_labels=["Available", "Missing"]
    )

    ax.set_title(battery_id)
    ax.set_ylabel("SoH (%)")
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("SoH Distribution by Impedance Availability")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=False)

for ax, battery_id in zip(axes, battery_ids):

    df = combined_df[
        combined_df["battery_id"] == battery_id
    ].copy()

    ax.scatter(
        range(1, len(df) + 1),
        df["impedance_available"],
        s=18
    )

    ax.set_title(battery_id)
    ax.set_ylabel("Impedance")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Missing", "Available"])
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Discharge Cycle")

plt.suptitle("Impedance Measurement Availability Across Battery Life")
plt.tight_layout()
plt.show()

## 5. Modelling Dataset Construction

Construct the complete discharge dataset and the impedance-available subset. Define the operational and impedance-enhanced feature sets and prepare the battery-wise datasets for cross-validation.

In [ ]:
# Dataset A: all discharge cycles
model_dataset_A = combined_df.copy()

# Dataset B: only cycles with valid impedance measurements
model_dataset_B = combined_df[
    combined_df["impedance_available"] == 1
].copy()

# Reset indices
model_dataset_A = model_dataset_A.reset_index(drop=True)
model_dataset_B = model_dataset_B.reset_index(drop=True)

# Verify dataset sizes
print("Dataset A — Voltage and Temperature")
print("Shape:", model_dataset_A.shape)

print("\nDataset B — Voltage, Temperature and Impedance")
print("Shape:", model_dataset_B.shape)

print("\nObservations per battery in Dataset B:")
print(model_dataset_B.groupby("battery_id").size())

# Verify no missing impedance values in Dataset B
print(
    "\nMissing Re values:",
    model_dataset_B["Re"].isna().sum()
)

print(
    "Missing Rct values:",
    model_dataset_B["Rct"].isna().sum()
)

In [ ]:
# Define the input features

features_A = [
    "voltage_mean",
    "voltage_min",
    "voltage_max",
    "current_mean",
    "temperature_mean",
    "temperature_max",
    "temperature_rise"
]

features_B = features_A + ["Re", "Rct"]


# Prepare Leave-One-Battery-Out datasets

loso_datasets = {}

for test_battery in battery_ids:

    # Dataset A: all discharge observations from other batteries
    train_A = model_dataset_A[
        model_dataset_A["battery_id"] != test_battery
    ].copy()

    # Dataset B: impedance-available observations from other batteries
    train_B = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    # Use the same matched test observations for both models
    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    loso_datasets[test_battery] = {
        "train_A": train_A,
        "train_B": train_B,
        "test": test_data
    }

    print(f"\nTest battery: {test_battery}")
    print("Dataset A training observations:", len(train_A))
    print("Dataset B training observations:", len(train_B))
    print("Common test observations:", len(test_data))

## 6. Cross-Battery SoH Modelling and Evaluation

Evaluate baseline and impedance-enhanced Ridge regression models using Leave-One-Battery-Out (LOBO) validation. The held-out battery is excluded from model training, and the same impedance-available test observations are used for fair model comparison.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

results = []

for test_battery in battery_ids:

    datasets = loso_datasets[test_battery]

    # Training data
    train_A = datasets["train_A"]
    train_B = datasets["train_B"]

    # Common test data
    test_data = datasets["test"]

    y_train_A = train_A["SoH_percent"]
    y_train_B = train_B["SoH_percent"]
    y_test = test_data["SoH_percent"]

    # Define models
    model_A = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    model_B = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    # Train Model A: voltage and temperature
    model_A.fit(train_A[features_A], y_train_A)

    # Train Model B: voltage, temperature and impedance
    model_B.fit(train_B[features_B], y_train_B)

    # Predictions on the same held-out battery observations
    predictions_A = model_A.predict(test_data[features_A])
    predictions_B = model_B.predict(test_data[features_B])

    # Calculate evaluation metrics
    for model_name, predictions in [
        ("Ridge - Voltage & Temperature", predictions_A),
        ("Ridge - With Impedance", predictions_B)
    ]:

        results.append({
            "test_battery": test_battery,
            "model": model_name,
            "observations": len(y_test),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, predictions)
            ),
            "R2": r2_score(y_test, predictions)
        })

# Create results table
loso_results = pd.DataFrame(results)

print("Leave-One-Battery-Out Results:")
display(
    loso_results.round(4)
)

In [ ]:
results_fair = []

for test_battery in battery_ids:

    # Use the same impedance-available training observations
    # for both models
    train_data = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    # Same held-out test observations
    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    y_train = train_data["SoH_percent"]
    y_test = test_data["SoH_percent"]

    # Model A: voltage and temperature
    model_A = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    # Model B: voltage, temperature and impedance
    model_B = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    model_A.fit(train_data[features_A], y_train)
    model_B.fit(train_data[features_B], y_train)

    predictions_A = model_A.predict(test_data[features_A])
    predictions_B = model_B.predict(test_data[features_B])

    for model_name, predictions in [
        ("Voltage & Temperature", predictions_A),
        ("Voltage, Temperature & Impedance", predictions_B)
    ]:

        results_fair.append({
            "test_battery": test_battery,
            "model": model_name,
            "observations": len(y_test),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, predictions)
            ),
            "R2": r2_score(y_test, predictions)
        })

fair_results_df = pd.DataFrame(results_fair)

display(fair_results_df.round(4))

In [ ]:
prediction_results = {}

for test_battery in battery_ids:

    train_data = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    y_train = train_data["SoH_percent"]
    y_test = test_data["SoH_percent"]

    # Model A: voltage and temperature
    model_A = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    # Model B: voltage, temperature and impedance
    model_B = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    model_A.fit(train_data[features_A], y_train)
    model_B.fit(train_data[features_B], y_train)

    predictions_A = model_A.predict(test_data[features_A])
    predictions_B = model_B.predict(test_data[features_B])

    prediction_results[test_battery] = pd.DataFrame({
        "actual_SoH": y_test.to_numpy(),
        "predicted_A": predictions_A,
        "predicted_B": predictions_B
    }).reset_index(drop=True)


# Plot predictions for each battery
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, battery_id in zip(axes.flatten(), battery_ids):

    results = prediction_results[battery_id]

    ax.plot(
        results["actual_SoH"].values,
        label="Actual SoH",
        linewidth=2
    )

    ax.plot(
        results["predicted_A"].values,
        label="Voltage & Temperature",
        linestyle="--"
    )

    ax.plot(
        results["predicted_B"].values,
        label="With Impedance",
        linestyle=":"
    )

    ax.set_title(battery_id)
    ax.set_xlabel("Test Observation")
    ax.set_ylabel("SoH (%)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Cross-Battery SoH Estimation: Actual vs Predicted")
plt.tight_layout()
plt.show()

In [ ]:
bias_results = []

for battery_id in battery_ids:

    results = prediction_results[battery_id]

    for model_name, prediction_column in [
        ("Voltage & Temperature", "predicted_A"),
        ("With Impedance", "predicted_B")
    ]:

        errors = (
            results[prediction_column] -
            results["actual_SoH"]
        )

        bias_results.append({
            "battery": battery_id,
            "model": model_name,
            "mean_error": errors.mean(),
            "median_error": errors.median(),
            "mean_absolute_error": errors.abs().mean(),
            "maximum_absolute_error": errors.abs().max()
        })

bias_df = pd.DataFrame(bias_results)

display(bias_df.round(3))

In [ ]:
baseline_results = []

for test_battery in battery_ids:

    # Same training and test population as the fair experiment
    train_data = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    y_test = test_data["SoH_percent"]

    # Baseline: predict the training-set mean SoH
    baseline_prediction = np.full(
        len(test_data),
        train_data["SoH_percent"].mean()
    )

    # Evaluate baseline
    baseline_results.append({
        "battery": test_battery,
        "training_mean_SoH": train_data["SoH_percent"].mean(),
        "baseline_MAE": mean_absolute_error(
            y_test, baseline_prediction
        ),
        "baseline_RMSE": np.sqrt(
            mean_squared_error(y_test, baseline_prediction)
        ),
        "baseline_R2": r2_score(
            y_test, baseline_prediction
        )
    })

baseline_df = pd.DataFrame(baseline_results)

display(baseline_df.round(4))

In [ ]:
# Merge baseline and Ridge results
comparison_df = fair_results_df.merge(
    baseline_df[["battery", "baseline_MAE"]],
    left_on="test_battery",
    right_on="battery"
)

# Calculate percentage improvement
comparison_df["improvement_vs_baseline_%"] = (
    (comparison_df["baseline_MAE"] - comparison_df["MAE"])
    / comparison_df["baseline_MAE"]
) * 100

# Display relevant columns
display(
    comparison_df[
        [
            "test_battery",
            "model",
            "baseline_MAE",
            "MAE",
            "improvement_vs_baseline_%"
        ]
    ].round(2)
)

# Calculate observation-weighted overall MAE
print("\nOverall weighted MAE:")

for model_name, group in comparison_df.groupby("model"):

    weighted_mae = np.average(
        group["MAE"],
        weights=group["observations"]
    )

    print(f"{model_name}: {weighted_mae:.3f}")

test_observation_counts = (
    model_dataset_B.groupby("battery_id").size().to_dict()
)

weighted_baseline = np.average(
    baseline_df["baseline_MAE"],
    weights=baseline_df["battery"].map(test_observation_counts)
)

print(f"Baseline: {weighted_baseline:.3f}")

In [ ]:
from sklearn.inspection import permutation_importance

importance_results = []

for test_battery in battery_ids:

    train_data = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    model_B = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    model_B.fit(
        train_data[features_B],
        train_data["SoH_percent"]
    )

    # Permutation importance on the held-out battery
    importance = permutation_importance(
        model_B,
        test_data[features_B],
        test_data["SoH_percent"],
        scoring="neg_mean_absolute_error",
        n_repeats=30,
        random_state=42
    )

    for feature, mean_importance, std_importance in zip(
        features_B,
        importance.importances_mean,
        importance.importances_std
    ):

        importance_results.append({
            "battery": test_battery,
            "feature": feature,
            "importance": mean_importance,
            "std": std_importance
        })

importance_df = pd.DataFrame(importance_results)

# Display results
display(
    importance_df.pivot(
        index="feature",
        columns="battery",
        values="importance"
    ).round(4)
)

In [ ]:
ablation_results = []

feature_sets = {
    "Voltage & Temperature": features_A,
    "Voltage & Temperature + Re": features_A + ["Re"],
    "Voltage & Temperature + Rct": features_A + ["Rct"],
    "Voltage & Temperature + Re + Rct": features_B
}

for test_battery in battery_ids:

    train_data = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    y_train = train_data["SoH_percent"]
    y_test = test_data["SoH_percent"]

    for model_name, selected_features in feature_sets.items():

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=1))
        ])

        model.fit(
            train_data[selected_features],
            y_train
        )

        predictions = model.predict(
            test_data[selected_features]
        )

        ablation_results.append({
            "battery": test_battery,
            "model": model_name,
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, predictions)
            ),
            "R2": r2_score(y_test, predictions)
        })

ablation_df = pd.DataFrame(ablation_results)

display(
    ablation_df.pivot(
        index="model",
        columns="battery",
        values="MAE"
    ).round(3)
)

In [ ]:
# Observation-weighted MAE across all held-out batteries

summary_ablation = (
    ablation_df
    .groupby("model")
    .apply(
        lambda group: np.average(
            group["MAE"],
            weights=[
                model_dataset_B.groupby("battery_id").size().loc[b]
                for b in group["battery"]
            ]
        ),
        include_groups=False
    )
    .reset_index(name="weighted_MAE")
    .sort_values("weighted_MAE")
)

display(summary_ablation.round(4))

In [ ]:
re_correlation_results = []

for battery_id in battery_ids:

    df = model_dataset_B[
        model_dataset_B["battery_id"] == battery_id
    ]

    pearson_corr = df["Re"].corr(
        df["SoH_percent"],
        method="pearson"
    )

    spearman_corr = df["Re"].corr(
        df["SoH_percent"],
        method="spearman"
    )

    re_correlation_results.append({
        "battery": battery_id,
        "observations": len(df),
        "Pearson_Re_SoH": pearson_corr,
        "Spearman_Re_SoH": spearman_corr
    })

re_correlation_df = pd.DataFrame(
    re_correlation_results
)

display(re_correlation_df.round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, battery_id in zip(axes.flatten(), battery_ids):

    df = model_dataset_B[
        model_dataset_B["battery_id"] == battery_id
    ]

    ax.scatter(
        df["Re"],
        df["SoH_percent"],
        alpha=0.7
    )

    ax.set_title(battery_id)
    ax.set_xlabel("Electrolyte resistance Re (Ohms)")
    ax.set_ylabel("State of Health (%)")
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Relationship between Electrolyte Resistance and Battery SoH",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, battery_id in zip(axes.flatten(), battery_ids):

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
    )

    ax2 = ax.twinx()

    line1 = ax.plot(
        df["record_index"],
        df["SoH_percent"],
        marker="o",
        markersize=3,
        label="SoH (%)"
    )

    line2 = ax2.plot(
        df["record_index"],
        df["Re"],
        marker="s",
        markersize=3,
        linestyle="--",
        label="Re (Ohms)"
    )

    ax.set_title(battery_id)
    ax.set_xlabel("Discharge cycle index")
    ax.set_ylabel("SoH (%)")
    ax2.set_ylabel("Re (Ohms)")

    lines = line1 + line2
    ax.legend(lines, [line.get_label() for line in lines],
              loc="best")

    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Evolution of Battery Health and Electrolyte Resistance",
    fontsize=14
)

plt.tight_layout()
plt.show()

## 7. Temporal and Statistical Diagnostics

Examine the temporal relationship between discharge-cycle progression, impedance resistance, and SoH. This section includes detrended correlations, regression diagnostics, multicollinearity assessment, and temporal train-test comparisons.

In [ ]:
from scipy.stats import pearsonr

adjusted_correlations_corrected = []

for battery_id in battery_ids:

    # Select impedance-matched observations
    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    # Use the original discharge-cycle indices
    cycle_number = df["record_index"].to_numpy()

    soh = df["SoH_percent"].to_numpy()
    re = df["Re"].to_numpy()

    # Fit linear trends against the actual cycle indices
    soh_trend = np.polyfit(cycle_number, soh, 1)
    re_trend = np.polyfit(cycle_number, re, 1)

    # Calculate residuals (variations not explained by cycle number)
    soh_residuals = (
        soh - np.polyval(soh_trend, cycle_number)
    )

    re_residuals = (
        re - np.polyval(re_trend, cycle_number)
    )

    # Calculate correlation between residuals
    corr, p_value = pearsonr(
        soh_residuals,
        re_residuals
    )

    adjusted_correlations_corrected.append({
        "battery": battery_id,
        "observations": len(df),
        "adjusted_correlation": corr,
        "p_value": p_value
    })

# Display results
adjusted_corr_corrected_df = pd.DataFrame(
    adjusted_correlations_corrected
)

display(adjusted_corr_corrected_df.round(4))

In [ ]:
import statsmodels.api as sm

regression_results = []

for battery_id in battery_ids:

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    y = df["SoH_percent"]

    # Model 1: Cycle number only
    X1 = sm.add_constant(
        df[["record_index"]]
    )
    model_1 = sm.OLS(y, X1).fit()

    # Model 2: Cycle number + Re
    X2 = sm.add_constant(
        df[["record_index", "Re"]]
    )
    model_2 = sm.OLS(y, X2).fit()

    regression_results.append({
        "battery": battery_id,
        "observations": len(df),
        "R2_cycle_only": model_1.rsquared,
        "R2_cycle_plus_Re": model_2.rsquared,
        "R2_improvement": (
            model_2.rsquared - model_1.rsquared
        ),
        "Re_coefficient": model_2.params["Re"],
        "Re_p_value": model_2.pvalues["Re"]
    })

regression_df = pd.DataFrame(regression_results)

display(regression_df.round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, battery_id in zip(axes.flatten(), battery_ids):

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    X = sm.add_constant(
        df[["record_index", "Re"]]
    )

    y = df["SoH_percent"]

    model = sm.OLS(y, X).fit()

    residuals = model.resid

    ax.scatter(
        df["record_index"],
        residuals,
        alpha=0.7
    )

    ax.axhline(0, linestyle="--")

    ax.set_title(battery_id)
    ax.set_xlabel("Discharge cycle index")
    ax.set_ylabel("Regression residuals (SoH percentage points)")
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Residual Analysis: Cycle Number + Re Regression",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

nonlinear_results = []

for battery_id in battery_ids:

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    X = df[["record_index", "Re"]]
    y = df["SoH_percent"]

    # Linear model
    linear_model = LinearRegression()
    linear_model.fit(X, y)

    pred_linear = linear_model.predict(X)

    # Polynomial model (degree 2)
    polynomial_model = Pipeline([
        ("poly", PolynomialFeatures(
            degree=2,
            include_bias=False
        )),
        ("regression", LinearRegression())
    ])

    polynomial_model.fit(X, y)

    pred_polynomial = polynomial_model.predict(X)

    nonlinear_results.append({
        "battery": battery_id,
        "Linear_MAE": mean_absolute_error(y, pred_linear),
        "Polynomial_MAE": mean_absolute_error(y, pred_polynomial),
        "Linear_R2": r2_score(y, pred_linear),
        "Polynomial_R2": r2_score(y, pred_polynomial)
    })

nonlinear_df = pd.DataFrame(nonlinear_results)

display(nonlinear_df.round(4))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

temporal_results = []

for battery_id in battery_ids:

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    # Chronological 70/30 split
    split_index = int(len(df) * 0.70)

    train = df.iloc[:split_index]
    test = df.iloc[split_index:]

    X_train = train[["record_index", "Re"]]
    X_test = test[["record_index", "Re"]]

    y_train = train["SoH_percent"]
    y_test = test["SoH_percent"]

    # Linear model
    linear_model = LinearRegression()
    linear_model.fit(X_train, y_train)

    pred_linear = linear_model.predict(X_test)

    # Polynomial model
    polynomial_model = Pipeline([
        ("poly", PolynomialFeatures(
            degree=2,
            include_bias=False
        )),
        ("regression", LinearRegression())
    ])

    polynomial_model.fit(X_train, y_train)

    pred_polynomial = polynomial_model.predict(X_test)

    temporal_results.append({
        "battery": battery_id,
        "train_observations": len(train),
        "test_observations": len(test),
        "Linear_MAE": mean_absolute_error(
            y_test, pred_linear
        ),
        "Polynomial_MAE": mean_absolute_error(
            y_test, pred_polynomial
        ),
        "Linear_R2": r2_score(
            y_test, pred_linear
        ),
        "Polynomial_R2": r2_score(
            y_test, pred_polynomial
        )
    })

temporal_df = pd.DataFrame(temporal_results)

display(temporal_df.round(4))

In [ ]:
distribution_results = []

for battery_id in battery_ids:

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(df) * 0.70)

    train = df.iloc[:split_index]
    test = df.iloc[split_index:]

    distribution_results.append({
        "battery": battery_id,
        "train_SoH_min": train["SoH_percent"].min(),
        "train_SoH_max": train["SoH_percent"].max(),
        "test_SoH_min": test["SoH_percent"].min(),
        "test_SoH_max": test["SoH_percent"].max()
    })

distribution_df = pd.DataFrame(distribution_results)

display(distribution_df.round(2))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

comparison_results = []

for battery_id in battery_ids:

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(df) * 0.70)

    train = df.iloc[:split_index]
    test = df.iloc[split_index:]

    y_train = train["SoH_percent"]
    y_test = test["SoH_percent"]

    # Model 1: Cycle only
    X1_train = train[["record_index"]]
    X1_test = test[["record_index"]]

    model_1 = LinearRegression()
    model_1.fit(X1_train, y_train)
    pred_1 = model_1.predict(X1_test)

    # Model 2: Cycle + Re
    X2_train = train[["record_index", "Re"]]
    X2_test = test[["record_index", "Re"]]

    model_2 = LinearRegression()
    model_2.fit(X2_train, y_train)
    pred_2 = model_2.predict(X2_test)

    # Model 3: Polynomial Cycle + Re
    model_3 = Pipeline([
        ("poly", PolynomialFeatures(
            degree=2,
            include_bias=False
        )),
        ("regression", LinearRegression())
    ])

    model_3.fit(X2_train, y_train)
    pred_3 = model_3.predict(X2_test)

    predictions = {
        "Cycle only": pred_1,
        "Cycle + Re": pred_2,
        "Polynomial": pred_3
    }

    for model_name, predictions_test in predictions.items():

        comparison_results.append({
            "battery": battery_id,
            "model": model_name,
            "MAE": mean_absolute_error(
                y_test, predictions_test
            ),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, predictions_test)
            ),
            "R2": r2_score(
                y_test, predictions_test
            )
        })

comparison_df = pd.DataFrame(comparison_results)

display(
    comparison_df.pivot(
        index="battery",
        columns="model",
        values="MAE"
    ).round(4)
)

In [ ]:
# Number of test observations per battery
test_weights = {}

for battery_id in battery_ids:
    battery_data = model_dataset_B[
        model_dataset_B["battery_id"] == battery_id
    ]
    split_index = int(len(battery_data) * 0.70)
    test_weights[battery_id] = len(battery_data) - split_index

# Add observation counts
comparison_df["test_observations"] = (
    comparison_df["battery"].map(test_weights)
)

# Calculate overall model performance
overall_results = (
    comparison_df
    .groupby("model")
    .apply(
        lambda group: pd.Series({
            "weighted_MAE": np.average(
                group["MAE"],
                weights=group["test_observations"]
            ),
            "mean_battery_MAE": group["MAE"].mean(),
            "weighted_RMSE": np.sqrt(
                np.average(
                    group["RMSE"] ** 2,
                    weights=group["test_observations"]
                )
            ),
            "total_test_observations":
                group["test_observations"].sum()
        }),
        include_groups=False
    )
    .sort_values("weighted_MAE")
)

display(overall_results.round(4))

In [ ]:
# Inspect the Cycle + Re model for each battery

selected_results = []

for battery_id in battery_ids:

    train_data = model_dataset_B[
        (model_dataset_B["battery_id"] == battery_id)
    ].copy()

    X = train_data[["record_index", "Re"]]
    y = train_data["SoH_percent"]

    model = LinearRegression()
    model.fit(X, y)

    selected_results.append({
        "battery": battery_id,
        "cycle_coefficient": model.coef_[0],
        "Re_coefficient": model.coef_[1],
        "intercept": model.intercept_
    })

coefficients_df = pd.DataFrame(selected_results)

display(coefficients_df.round(4))

In [ ]:
# Calculate coefficients using only the temporal training data

training_coefficients = []

for battery_id in battery_ids:

    battery_data = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(battery_data) * 0.7)
    train_data = battery_data.iloc[:split_index]

    X_train = train_data[["record_index", "Re"]]
    y_train = train_data["SoH_percent"]

    model = LinearRegression()
    model.fit(X_train, y_train)

    training_coefficients.append({
        "battery": battery_id,
        "training_observations": len(train_data),
        "cycle_coefficient": model.coef_[0],
        "Re_coefficient": model.coef_[1],
        "intercept": model.intercept_
    })

training_coefficients_df = pd.DataFrame(training_coefficients)

display(training_coefficients_df.round(4))

In [ ]:
# Check correlation between cycle number and Re

cycle_re_correlations = []

for battery_id in battery_ids:

    battery_data = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(battery_data) * 0.7)
    train_data = battery_data.iloc[:split_index]

    correlation = train_data[
        ["record_index", "Re"]
    ].corr().iloc[0, 1]

    cycle_re_correlations.append({
        "battery": battery_id,
        "training_observations": len(train_data),
        "cycle_Re_correlation": correlation
    })

cycle_re_corr_df = pd.DataFrame(cycle_re_correlations)

display(cycle_re_corr_df.round(4))

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF for each battery's training data

vif_results = []

for battery_id in battery_ids:

    battery_data = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(battery_data) * 0.7)
    train_data = battery_data.iloc[:split_index]

    X = train_data[["record_index", "Re"]].copy()

    # Standardise variables for numerical stability
    X = (X - X.mean()) / X.std()

    vif_cycle = variance_inflation_factor(X.values, 0)
    vif_re = variance_inflation_factor(X.values, 1)

    vif_results.append({
        "battery": battery_id,
        "VIF_cycle": vif_cycle,
        "VIF_Re": vif_re
    })

vif_df = pd.DataFrame(vif_results)

display(vif_df.round(2))

In [ ]:
# Test whether resistance deviations add information
# beyond the general cycle-related resistance trend.

from sklearn.linear_model import LinearRegression

residual_results = []

for battery_id in battery_ids:

    battery_data = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(battery_data) * 0.7)

    train_data = battery_data.iloc[:split_index].copy()
    test_data = battery_data.iloc[split_index:].copy()

    # Step 1: Estimate expected Re from cycle number
    resistance_trend = LinearRegression()

    resistance_trend.fit(
        train_data[["record_index"]],
        train_data["Re"]
    )

    # Step 2: Calculate resistance residuals
    train_data["Re_residual"] = (
        train_data["Re"] -
        resistance_trend.predict(train_data[["record_index"]])
    )

    test_data["Re_residual"] = (
        test_data["Re"] -
        resistance_trend.predict(test_data[["record_index"]])
    )

    # Step 3: Fit SoH models
    model_cycle = LinearRegression()
    model_residual = LinearRegression()

    model_cycle.fit(
        train_data[["record_index"]],
        train_data["SoH_percent"]
    )

    model_residual.fit(
        train_data[["record_index", "Re_residual"]],
        train_data["SoH_percent"]
    )

    # Step 4: Evaluate on the later observations
    pred_cycle = model_cycle.predict(
        test_data[["record_index"]]
    )

    pred_residual = model_residual.predict(
        test_data[["record_index", "Re_residual"]]
    )

    residual_results.append({
        "battery": battery_id,
        "test_observations": len(test_data),
        "Cycle_only_MAE": mean_absolute_error(
            test_data["SoH_percent"], pred_cycle
        ),
        "Cycle_plus_residual_Re_MAE": mean_absolute_error(
            test_data["SoH_percent"], pred_residual
        )
    })

residual_comparison_df = pd.DataFrame(residual_results)

display(residual_comparison_df.round(4))

## 8. Results Export

Optionally connect Google Drive to save the generated modelling tables, the combined dataset, and a reproducible modelling summary. The analysis itself is performed before this persistence section.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

project_folder = "/content/drive/MyDrive/Battery_Digital_Twin"

results_folder = os.path.join(project_folder, "results")

os.makedirs(results_folder, exist_ok=True)

print("Project folder:", project_folder)
print("Results folder:", results_folder)
print("Folder created successfully!")

In [ ]:
import os

# Results to export
results_to_save = {
    "temporal_model_comparison": comparison_df,
    "overall_model_performance": overall_results,
    "full_data_coefficients": coefficients_df,
    "training_coefficients": training_coefficients_df,
    "cycle_Re_correlations": cycle_re_corr_df,
    "multicollinearity_VIF": vif_df,
    "residual_Re_comparison": residual_comparison_df
}

# Save each result as a CSV file
for filename, dataframe in results_to_save.items():
    filepath = os.path.join(
        results_folder,
        f"{filename}.csv"
    )
    dataframe.to_csv(filepath, index=True)
    print("Saved:", filename + ".csv")

print("\nAll modelling results exported successfully!")

In [ ]:
# Build the modelling summary from the current notebook results.
# This avoids retaining fixed metrics that can become stale after re-running the analysis.

total_discharge_observations = len(combined_df)
matched_impedance_observations = int(combined_df["impedance_available"].sum())

summary_lines = [
    "# Battery Digital Twin – Modelling Results",
    "",
    "## 1. Dataset",
    "- Source: NASA Battery Aging Dataset",
    f"- Batteries: {', '.join(battery_ids)}",
    f"- Total discharge observations: {total_discharge_observations}",
    f"- Matched impedance observations: {matched_impedance_observations}",
    "- Health target: State of Health (SoH), calculated from discharge capacity relative to the initial measured capacity.",
    "",
    "## 2. Modelling objective",
    "Investigate whether electrical and thermal indicators, particularly impedance resistance (Re), provide useful information for battery SoH estimation.",
    "",
    "## 3. Temporal model comparison",
    "The following metrics are calculated from the current temporal train-test evaluation:",
    "",
    overall_results.reset_index().to_string(index=False),
    "",
    "## 4. Cycle–resistance relationship",
    cycle_re_corr_df.to_string(index=False),
    "",
    "## 5. Multicollinearity diagnostics",
    vif_df.to_string(index=False),
    "",
    "## 6. Residual resistance experiment",
    residual_comparison_df.to_string(index=False),
    "",
    "## 7. Interpretation and limitations",
    "- Cycle progression provides a temporal indicator of degradation.",
    "- The contribution of impedance resistance should be interpreted alongside its correlation with cycle progression and the VIF diagnostics.",
    "- Results are battery-dependent and should not be interpreted as universal performance guarantees.",
    "- The dataset includes only four battery cells; B0018 has fewer observations than the other cells.",
    "- The temporal evaluation is an empirical forecasting experiment, not a validated electrochemical model.",
    "- The current estimator uses cycle-level measurements after discharge, not pre-cycle forecasting.",
    "",
    "## 8. Next development stage",
    "Connect the validated data-processing and estimation components to the Digital Twin state representation, update logic, and monitoring interface."
]

report = "\n".join(summary_lines)

report_path = os.path.join(results_folder, "modelling_summary.md")
with open(report_path, "w", encoding="utf-8") as file:
    file.write(report)

print("Dynamic modelling summary saved:", report_path)

In [ ]:
# Export the combined battery dataset

combined_data_path = os.path.join(
    results_folder,
    "combined_battery_dataset.csv"
)

combined_df.to_csv(combined_data_path, index=False)

print("Dataset exported successfully!")
print("File:", combined_data_path)
print("Shape:", combined_df.shape)